# Hyper Parameter Tuning (Optimization)


Every machine learning system has hyperparameters, hyperparameter optimization (HPO) aims at setting the "best" hyperparameters to optimize model performance.



## 1 The mathematical setup

Let $\mathcal{A}$ denote a machine learning algorithm with $N$ hyperparameters. We denote the space of the n-th hyperparameter by $\Lambda_n$ and the overall hyperparameter configuration space as $ \Lambda = \Lambda_1 \times \dots \times \Lambda_N$ . A vector of hyperparameters is denoted by $λ ∈ \Lambda$, and $\mathcal{A}$ with its hyperparameters instantiated to $\lambda$ is denoted by $A_{\lambda}$. Given a data set $\mathcal{D}$, our goal is to find

$$\lambda^* = arg\min_{\lambda\in\Lambda} E_{(D_{train},D_{valid})}[V(A_{\lambda},D_{train},D_{valid})]$$

where $V(\cdot)$ is an evaluation metric for the model performance.





## 2 Model-free blackbox optimization methods


### 2.1 Grid search
Grid search is the most basic HPO method. Grid search asks user to specify a finite set of values for each hyperparameter, and it evaluates the Cartesian product of these sets. This suffers from the curse of dimensionality since the required number of function evaluations grows exponentially with the dimensionality of the configuration space. An additional problem of grid search is that increasing the resolution of discretization substantially increases the required number of function evaluations.

As an example, let's train decision trees with various `max_depth` and `criterion` and evaluate their cross-validation accuracy in distingurishing Iris-Virginica. For this task, we can use Scikit-Learn's `GridSearchCV`.

In [1]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
import pandas as pd

X,y = load_iris(return_X_y=True)

param_grid = {
    'max_depth':[1,2,3,4,5],
    'criterion':['gini','entropy']
    }

grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X,y)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [1, 2, 3, 4, 5]},
             scoring='accuracy')

In [4]:
from sklearn.metrics import f1_score, make_scorer
f1_score = make_scorer(f1_score, average='weighted')
grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5, scoring=f1_score)
grid_search.fit(X,y)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [1, 2, 3, 4, 5]},
             scoring=make_scorer(f1_score, response_method='predict', average=weighted))

This param_grid tells Scikit-Learn to explore all 5*2 = 10 combinations of `max_depth` and `criterion` hyperparameter values specified in the dict. It will train each model five times (since we are using five-fold cross validation) and evaluate the performance on the cross-validation data. It may take quite a long time, but when it is done you can get the best combination of parameters like this

In [5]:
grid_search.best_params_

{'criterion': 'gini', 'max_depth': 4}

You can also get the best estimator directly

In [6]:
grid_search.best_estimator_

DecisionTreeClassifier(max_depth=4)

And of course the evaluation scores are also available

In [7]:
pd.DataFrame(grid_search.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001265,0.000605,0.002913,0.001195,gini,1,"{'criterion': 'gini', 'max_depth': 1}",0.555556,0.555556,0.555556,0.555556,0.555556,0.555556,0.000000,9
1,0.000939,0.000036,0.002187,0.000023,gini,2,"{'criterion': 'gini', 'max_depth': 2}",0.933333,0.966583,0.899749,0.866667,1.000000,0.933266,0.047164,7
2,0.000960,0.000036,0.002202,0.000046,gini,3,"{'criterion': 'gini', 'max_depth': 3}",0.966583,0.966583,0.932660,0.933333,1.000000,0.959832,0.025080,3
3,0.001020,0.000091,0.002260,0.000087,gini,4,"{'criterion': 'gini', 'max_depth': 4}",0.966583,0.966583,0.899749,1.000000,1.000000,0.966583,0.036606,1
4,0.000967,0.000030,0.002186,0.000037,gini,5,"{'criterion': 'gini', 'max_depth': 5}",0.966583,0.966583,0.899749,0.966583,1.000000,0.959900,0.032742,2
5,0.000923,0.000050,0.002182,0.000058,entropy,1,"{'criterion': 'entropy', 'max_depth': 1}",0.555556,0.555556,0.555556,0.555556,0.555556,0.555556,0.000000,9
6,0.001032,0.000148,0.002388,0.000396,entropy,2,"{'criterion': 'entropy', 'max_depth': 2}",0.933333,0.966583,0.899749,0.866667,1.000000,0.933266,0.047164,7
7,0.001030,0.000016,0.002284,0.000109,entropy,3,"{'criterion': 'entropy', 'max_depth': 3}",0.966583,0.966583,0.932660,0.932660,1.000000,0.959697,0.025224,4
8,0.000991,0.000012,0.002200,0.000041,entropy,4,"{'criterion': 'entropy', 'max_depth': 4}",0.966583,0.966583,0.899749,0.933333,1.000000,0.953250,0.034059,5
9,0.000977,0.000027,0.002176,0.000058,entropy,5,"{'criterion': 'entropy', 'max_depth': 5}",0.966583,0.966583,0.899749,0.933333,1.000000,0.953250,0.034059,5


### 2.2 Random search
A simple alternative to grid search is random search. As the name suggests, random search samples configurations at random until a certain budget for the search is exhausted. This works better than grid search when the search space is large.

Further advantages over grid search include easier parallelization (since workers do not need to communicate with each other and failing workers do not leave holes in the design) and flexible resource allocation (since one can add an arbitrary number of random points to a random search design to still yield a random search design; the equivalent does not hold for grid search).



In [9]:
from sklearn.model_selection import RandomizedSearchCV
from scipy import stats

param_dist = {
    "max_depth": stats.randint(1,10),
    "ccp_alpha": stats.uniform(0,10) #Distributions must provide a rvs method for sampling (such as those from scipy.stats.distributions)
}

rand_search = RandomizedSearchCV(DecisionTreeClassifier(), param_dist, n_iter=10, cv=5, scoring='accuracy', random_state=1)
rand_search.fit(X,y)

RandomizedSearchCV(cv=5, estimator=DecisionTreeClassifier(),
                   param_distributions={'ccp_alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c5fcb5c1bb0>,
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c5fcba70f80>},
                   random_state=1, scoring='accuracy')

Similar to GridSearchCV, we can explore the best parameters, scores, and estimator.

In [10]:
rand_search.best_params_

{'ccp_alpha': np.float64(0.0011437481734488664), 'max_depth': 6}

In [11]:
rand_search.best_score_

np.float64(0.9666666666666668)

In [12]:
rand_search.best_estimator_

DecisionTreeClassifier(ccp_alpha=np.float64(0.0011437481734488664), max_depth=6)

### 2.3 Population-based methods

Population-based methods, such as *evolutionary algorithms*, are optimization algorithms that iteratively improve a population of configurations by applying local perturbations (so-called mutations) and combinations of different members (so-called crossover) to obtain a new generation of better configurations.

One of the best known population-based methods is the covariance matrix adaption evolutionary strategy (CMA-ES); this simple evolutionary strategy samples configurations from a N-variate Gaussian whose mean and covariance are updated in each generation based on the "survivors".

## 3 Bayesian optimization

Bayesian optimization is a state-of-the-art optimization framework for the global optimization of expensive blackbox functions, such as $V(\lambda)$, which recently gained traction in HPO. For an in-depth introduction to Bayesian optimization, check the excellent tutorials by [Shahriari et al.](https://ieeexplore.ieee.org/document/7352306) and [Brochu et al.](https://arxiv.org/pdf/1012.2599.pdf).

In general, we prescribe a prior belief over the possible objective functions and then sequentially refine this model as data are observed via Bayesian posterior updating.

This method is implemented in a separate package called `scikit-optimize`.

In [13]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.8 MB/s eta 0:00:00


In [14]:
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn.svm import SVC
import numpy as np

bayes_search = BayesSearchCV(
    SVC(),
    {
      'C': Real(1e-6, 1e+6, prior='log-uniform'),
      'gamma': Real(1e-6, 1e+1, prior='log-uniform'),
      'degree': Integer(1,8),
      'kernel': Categorical(['linear', 'poly', 'rbf']),
    },
    n_iter=10,
    cv=5,
    scoring='accuracy',
    random_state=1
)
np.int = int # there is a bug in skopt waiting to be fixed

bayes_search.fit(X, y)

BayesSearchCV(cv=5, estimator=SVC(), n_iter=10, random_state=1,
              scoring='accuracy',
              search_spaces={'C': Real(low=1e-06, high=1000000.0, prior='log-uniform', transform='normalize'),
                             'degree': Integer(low=1, high=8, prior='uniform', transform='normalize'),
                             'gamma': Real(low=1e-06, high=10.0, prior='log-uniform', transform='normalize'),
                             'kernel': Categorical(categories=('linear', 'poly', 'rbf'), prior=None)})

In [15]:
bayes_search.best_params_

OrderedDict([('C', 62.605882017291876),
             ('degree', 2),
             ('gamma', 0.016313171363561894),
             ('kernel', 'poly')])

In [16]:
bayes_search.best_score_

np.float64(0.9800000000000001)

## 4 Multi-fidelity optimization

Increasing dataset sizes and increasingly complex models are major hurdles in HPO. A common technique to speed up manual tuning is therefore to probe an algorithm/hyperparameter configuration on a small subset of the data, by training it only for a few iterations, by running it on a subset of features, by only using one or a few of the cross-validation folds, or by using down-sampled images in computer vision. Multi-fidelity methods cast such manual heuristics into formal algorithms, using so-called low fidelity approximations of the actual loss function to minimize. These approximations introduce a tradeoff between optimization performance and runtime, but in practice, the obtained speedups often outweigh the approximation error.

One of the simplest example of multi-fidelity optimization is successive halving. The basic idea is to start with the a large parameter space, and then try each of these candidate on a `min_resources` - for example, the resource maybe defined as the number of observations. We then discard a fraction of the worst performing candidates and train the remaining ones with more resources. Iterating this process, until at least one candidate reaches `max_resources`.

Scikit-learn implements this method by `HalvingGridSearchCV` and `HalvingRandomSearchCV`

In [17]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV


halv_rand_search = HalvingRandomSearchCV(
    DecisionTreeClassifier(),
    {
      "max_depth": stats.randint(1,10),
      "ccp_alpha": stats.uniform(0,10)
    },
    n_candidates=1000,
    resource = 'n_samples',
    max_resources = 'auto',
    min_resources = 10,
    cv=5,
    scoring='accuracy',
    random_state=1
    )
halv_rand_search.fit(X,y)

HalvingRandomSearchCV(estimator=DecisionTreeClassifier(), min_resources=10,
                      n_candidates=1000,
                      param_distributions={'ccp_alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7c5fcb2c9610>,
                                           'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c5fcd80cbc0>},
                      random_state=1, scoring='accuracy')

In [18]:
halv_rand_search.best_score_

np.float64(0.9555555555555555)